# Rețele neuronale și deep learning — Seminarul 5: Rețele recurente: RNN, LSTM și GRU pentru secvențe și serii de timp financiare
Prof. dr. Daniel Traian Pele — Academia de Studii Economice din București (ASE); Academia Română, Institutul de Prognoză Economică (IPE)

[Deschide în Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L5_seminar_ro.ipynb)

Sarcinile seminarului, pe datele reale din `../data/clean` (vezi `../data/SOURCES.md`). Scrieți codul în celulele goale de sub fiecare sarcină; păstrați valoarea fixă a generatorului aleator, ca rezultatele să poată fi comparate.

In [ ]:
# Google Colab: copiem repo-ul cursului, ca datele și rezultatele salvate să fie găsite
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Pregătire

In [ ]:
import torch, torch.nn as nn
torch.set_num_threads(4)
x = pd.read_csv(DATA + 'markets_daily.csv', parse_dates=['date'])
def ohlc(tk):
    d = x[x.ticker == tk].set_index('date')[['open', 'high', 'low', 'close', 'adjclose']].dropna()
    end = (d.index.max() + pd.Timedelta(days=1)).to_period('M').to_timestamp() - pd.Timedelta(days=1)
    d = d.loc[:end]; d = d[d.high > d.low]
    gk = 0.5 * np.log(d.high / d.low) ** 2 - (2 * np.log(2) - 1) * np.log(d.close / d.open) ** 2
    d['y'] = np.log(gk.clip(lower=1e-8)); d['r'] = 100 * np.log(d.adjclose).diff()
    return d.dropna()
spx = ohlc('SP500')
TR_END, VA_END = pd.Timestamp('2016-12-31'), pd.Timestamp('2019-12-31')
print(spx.index.min().date(), spx.index.max().date(), len(spx))

In [ ]:
class RecNet(nn.Module):
    def __init__(self, cell, n_in, hidden=32, n_out=1, n_static=0):
        super().__init__()
        self.rnn = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[cell](n_in, hidden, batch_first=True)
        self.out = nn.Linear(hidden + n_static, n_out)
    def forward(self, x, s=None):
        h, _ = self.rnn(x)
        z = h[:, -1] if s is None else torch.cat([h[:, -1], s], 1)
        return self.out(z)

def train_rec(cell, Xtr, Ytr, Xva, Yva, hidden=32, epochs=40, lr=1e-3, bs=64, Str=None, Sva=None, patience=6, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    n_out = Ytr.shape[1] if Ytr.ndim > 1 else 1
    m = RecNet(cell, Xtr.shape[2], hidden, n_out, 0 if Str is None else Str.shape[1])
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    Xt, Yt = torch.tensor(Xtr), torch.tensor(Ytr).view(len(Ytr), -1)
    Xv, Yv = torch.tensor(Xva), torch.tensor(Yva).view(len(Yva), -1)
    St = None if Str is None else torch.tensor(Str); Sv = None if Sva is None else torch.tensor(Sva)
    g = torch.Generator().manual_seed(seed); best, state, bad = np.inf, None, 0
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(Xt), generator=g)
        for j in range(0, len(perm), bs):
            b = perm[j:j + bs]
            loss = ((m(Xt[b], None if St is None else St[b]) - Yt[b]) ** 2).mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        m.eval()
        with torch.no_grad():
            vl = float(((m(Xv, Sv) - Yv) ** 2).mean())
        if vl < best - 1e-5: best, bad, state = vl, 0, {k: v.clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    m.load_state_dict(state); m.eval()
    return m

def predict(m, X, S=None):
    with torch.no_grad():
        return m(torch.tensor(X), None if S is None else torch.tensor(S)).numpy()

def dm_test(l1, l2):
    from scipy.stats import norm
    d = np.asarray(l1) - np.asarray(l2); n = len(d); dc = d - d.mean(); L = int(4 * (n / 100) ** (2 / 9))
    v = dc @ dc / n + sum(2 * (1 - k / (L + 1)) * (dc[k:] @ dc[:-k]) / n for k in range(1, L + 1))
    s = d.mean() / np.sqrt(v / n); return round(float(s), 2), round(float(2 * (1 - norm.cdf(abs(s)))), 3)
def vol_data(d, W=22):
    y = d.y.values.astype(np.float32); r = d.r.values.astype(np.float32); tr = d.index <= TR_END
    mu, sd, sr = y[tr].mean(), y[tr].std(), r[tr].std()
    f = np.stack([(y - mu) / sd, r / sr, np.abs(r) / sr], 1).astype(np.float32)
    T_ = np.arange(W - 1, len(y) - 1)
    X = np.stack([f[t - W + 1:t + 1] for t in T_]); Y = ((y[T_ + 1] - mu) / sd).astype(np.float32)
    td = d.index[T_ + 1]
    H = np.column_stack([np.ones(len(T_)), y[T_], pd.Series(y).rolling(5).mean().values[T_], pd.Series(y).rolling(22).mean().values[T_]])
    return X, Y, td, y[T_ + 1], y[T_], H, mu, sd

**Sarcina 1.** Numărați parametrii unui strat LSTM și ai unui strat GRU cu $d=3$ și $n_h=32$, cu formulele $4n_h(n_h+d+1)$ și $3n_h(n_h+d+1)$, și comparați cu PyTorch (care păstrează doi vectori de termeni liberi).

In [ ]:
# codul vostru aici

**Sarcina 2.** Construiți log-varianța Garman–Klass a indicelui DAX (`ohlc('DAX')`) și comparați autocorelația sa de ordinul 1 cu cea a S&P 500.

In [ ]:
# codul vostru aici

**Sarcina 3.** Estimați modelul HAR pe anii de antrenare ai DAX (pînă în 2016) și raportați MSE pe test (din 2020) față de prognoza naivă.

In [ ]:
# codul vostru aici

**Sarcina 4.** Antrenați un GRU cu 32 de unități pe aceleași ferestre (oprire timpurie pe 2017–2019). Raportați MSE pe test pentru trei semințe.

In [ ]:
# codul vostru aici

**Sarcina 5.** Testați cu Diebold–Mariano dacă GRU mediat pe semințe depășește HAR pe DAX. Ce model ați pune în producție?

In [ ]:
# codul vostru aici

**Sarcina 6.** Consumul din România: calculați MAPE a prognozei naive sezoniere (aceeași oră cu o săptămînă înainte) pe zile ale săptămînii, în 2024–2026. Care zile sînt cele mai dificile și de ce?

In [ ]:
# codul vostru aici